---
title: "Stage 1: Collaboration And Configuration"
author: "Abdelhamed Nour"
description: "A second researcher changes the experiment without editing internals"
format:
  html:
    toc: true
    code-fold: false
    code-tools: true
execute:
  eval: false
  echo: true
---


In [ ]:
#| echo: false

#| default_exp workflow

In [ ]:
#| echo: false

#| export
from ipcv._workflow_impl import (
    SyntheticSegmentationDataset,
    TinySegmentationModel,
    TorchvisionSegmentationModel,
    TrainingConfig,
    VOCSegmentationTransform,
    build_dataloader,
    build_dataset,
    build_model,
    build_segmentation_preview_image,
    build_tiny_model,
    build_torchvision_model,
    build_validation_dataloader,
    build_validation_dataset,
    cli_main,
    console_log,
    dataloader_batch_count,
    dataloader_sample_count,
    evaluate_one_epoch,
    load_config,
    log_pytorch_model,
    log_segmentation_preview,
    mask_to_rgb_image,
    mlflow_input_example,
    mlflow_safe_params,
    run_training,
    sanitize_mask,
    save_checkpoint,
    segmentation_stats_to_metrics,
    set_seed,
    tensor_to_rgb_image,
    train_one_epoch,
    update_segmentation_stats,
    voc_color_palette,
)

__all__ = [
    "SyntheticSegmentationDataset",
    "TinySegmentationModel",
    "TorchvisionSegmentationModel",
    "TrainingConfig",
    "VOCSegmentationTransform",
    "build_dataloader",
    "build_dataset",
    "build_model",
    "build_segmentation_preview_image",
    "build_tiny_model",
    "build_torchvision_model",
    "build_validation_dataloader",
    "build_validation_dataset",
    "cli_main",
    "console_log",
    "dataloader_batch_count",
    "dataloader_sample_count",
    "evaluate_one_epoch",
    "load_config",
    "log_pytorch_model",
    "log_segmentation_preview",
    "mask_to_rgb_image",
    "mlflow_input_example",
    "mlflow_safe_params",
    "run_training",
    "sanitize_mask",
    "save_checkpoint",
    "segmentation_stats_to_metrics",
    "set_seed",
    "tensor_to_rgb_image",
    "train_one_epoch",
    "update_segmentation_stats",
    "voc_color_palette",
]

## A Collaborator Joins

A second researcher wants to try another run:

- fewer images for a smoke test
- a different image size
- a different checkpoint path
- maybe a smaller test model

They should not need to edit the training loop to do that.

::: {.notes}
Timing: 5 minutes. This is the central productification move: separate experiment choices from workflow code.
:::

## Factor The Notebook Into Parts

```{mermaid}
flowchart LR
  A[Notebook cells] --> B[Config]
  A --> C[Data builder]
  A --> D[Model builder]
  A --> E[Train one epoch]
  A --> F[Checkpoint writer]
  B --> G[train.py runner]
  C --> G
  D --> G
  E --> G
  F --> G
```

## The Configuration Becomes The Contract

```yaml
data:
  root: data
  image_size: 128
  max_samples: 12
model:
  model_name: unet
  encoder_name: resnet18
training:
  epochs: 1
  max_batches: 4
```

A new run is now a config change, not a rewrite.

## `train.py` Is Agnostic

```python
from ipcv.workflow import cli_main

if __name__ == "__main__":
    raise SystemExit(cli_main())
```

The runner knows how to start the workflow. It does not know VOC details, model internals, or notebook state.

## The Demo Commands

```bash
uv run python train.py --config params.yaml
uv run python train.py --config params.yaml --max-samples 4 --max-batches 1
uv run python train.py --config params.yaml --synthetic --model tiny --no-mlflow
```

The same entry point works for real VOC training and fast synthetic checks.

## What Changed

- The notebook still teaches the idea
- The package holds reusable behavior
- The YAML file holds choices
- The runner gives collaborators one stable command